In [1]:
# Setting Up the Environment

import sys
import os
import time
import logging
import torch
import numpy as np
from PIL import Image
import rembg
import pymeshlab as pymesh
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
from tkinter import Tk
from tkinter.filedialog import askopenfilename
from IPython.display import Video


In [2]:
# Configure Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")


In [3]:
# Timer Utility Class
class Timer:
    def __init__(self):
        self.items = {}
        self.time_scale = 1000.0  # ms
        self.time_unit = "ms"
    
    def start(self, name: str) -> None:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.items[name] = time.time()
        logging.info(f"{name} starting...")

    def end(self, name: str) -> float:
        if name not in self.items:
            return
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = self.items.pop(name)
        delta = time.time() - start_time
        t = delta * self.time_scale
        logging.info(f"{name} finished in {t:.2f}{self.time_unit}.")
        return t

timer = Timer()


In [4]:
# Model and Parameters Setup
device = "cuda:0" if torch.cuda.is_available() else "cpu"
pretrained_model_name_or_path = "stabilityai/TripoSR"
chunk_size = 8192
foreground_ratio = 0.85
output_dir = "output"
model_save_format = "obj"
render = True
os.makedirs(output_dir, exist_ok=True)


In [5]:
# Initialize TripoSR Model
timer.start("Initializing model")
model = TSR.from_pretrained(
    pretrained_model_name_or_path,
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(chunk_size)
model.to(device)
timer.end("Initializing model")


2025-02-08 15:46:35,833 [INFO] Initializing model starting...
2025-02-08 15:46:57,206 [INFO] Initializing model finished in 21373.43ms.


21373.433589935303

In [6]:
# Load Image
file_path = "examples/data_set/IMG_20180810_114846 - Copy.jpg"  # Change to your file path
original_image = Image.open(file_path).convert("RGBA")
original_image = original_image.resize((512, 512))


In [7]:
from pathlib import Path
# Create a new directory for this image's output
base_name = Path(file_path).stem  # Get the base name of the file
image_dir = os.path.join(output_dir, base_name)
os.makedirs(image_dir, exist_ok=True)


In [8]:
# Save the resized original image in the new directory
original_image.save(os.path.join(image_dir, "input_resized.png"))


In [9]:
# Process Image
timer.start("Processing image")
rembg_session = rembg.new_session()
image = remove_background(original_image, rembg_session)

2025-02-08 15:47:10,824 [INFO] Processing image starting...


In [10]:
# Resize Foreground
image = resize_foreground(image, foreground_ratio)

In [11]:
# Ensure Image Mode is Correct
if image.mode != "RGBA":
    image = image.convert("RGBA")

image = np.array(image).astype(np.float32) / 255.0
image = image[:, :, :3] * image[:, :, 3:4] + (1 - image[:, :, 3:4]) * 0.5
image = Image.fromarray((image * 255.0).astype(np.uint8))


In [12]:
# Save Processed Image
image.save(os.path.join(image_dir, "input_processed.png"))
timer.end("Processing image")

2025-02-08 15:47:20,469 [INFO] Processing image finished in 9644.78ms.


9644.780397415161

In [13]:
import imageio
print(imageio.plugins)


<module 'imageio.plugins' from 'c:\\Users\\akidu\\anaconda3\\envs\\newenv\\lib\\site-packages\\imageio\\plugins\\__init__.py'>


In [14]:
import imageio
import numpy as np

# Generate test frames
test_frames = [np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8) for _ in range(30)]

# Write video using FFmpeg plugin
with imageio.get_writer("test.mp4", fps=30, codec="libx264") as writer:
    for frame in test_frames:
        writer.append_data(frame)

print("Video created successfully.")


Video created successfully.


In [15]:
# Generate 3D Model and Render
timer.start("Running model")
with torch.no_grad():
    scene_codes = model([image], device=device)
timer.end("Running model")
logging.info("Model running complete.")

if render:
    timer.start("Rendering")
    logging.info("Starting rendering phase.")
    render_images = model.render(scene_codes, n_views=30, return_type="pil")

    # Save render images
    for ri, render_image in enumerate(render_images[0]):
        render_image.save(os.path.join(image_dir, f"render_{ri:03d}.png"))
        logging.info("Saving render image %d to %s", ri, render_image)


    # Save video of renders
    def save_video(frames, output_path, fps):
        import imageio
        import numpy as np
        with imageio.get_writer(output_path, fps=fps, codec="libx264") as writer:
            for frame in frames:
                # Convert PIL.Image to numpy array
                frame_array = np.array(frame)
                writer.append_data(frame_array)

    save_video(render_images[0], os.path.join(image_dir, "render.mp4"), fps=30)
    timer.end("Rendering")
    logging.info("Rendering complete.")


# Exporting Mesh
timer.start("Exporting mesh")
meshes = model.extract_mesh(scene_codes, has_vertex_color=False)
mesh_file = os.path.join(image_dir, f"mesh.{model_save_format}")
meshes[0].export(mesh_file)
timer.end("Exporting mesh")

logging.info("Processing complete.")


2025-02-08 15:47:31,205 [INFO] Running model starting...
2025-02-08 15:47:59,606 [INFO] Running model finished in 28400.95ms.
2025-02-08 15:47:59,606 [INFO] Model running complete.
2025-02-08 15:47:59,606 [INFO] Rendering starting...
2025-02-08 15:47:59,606 [INFO] Starting rendering phase.
2025-02-08 16:11:40,709 [INFO] Saving render image 0 to <PIL.Image.Image image mode=RGB size=256x256 at 0x230AE368220>
2025-02-08 16:11:40,749 [INFO] Saving render image 1 to <PIL.Image.Image image mode=RGB size=256x256 at 0x230AE368100>
2025-02-08 16:11:41,124 [INFO] Saving render image 2 to <PIL.Image.Image image mode=RGB size=256x256 at 0x230AE368160>
2025-02-08 16:11:41,154 [INFO] Saving render image 3 to <PIL.Image.Image image mode=RGB size=256x256 at 0x230AE368280>
2025-02-08 16:11:41,174 [INFO] Saving render image 4 to <PIL.Image.Image image mode=RGB size=256x256 at 0x230AE3682E0>
2025-02-08 16:11:41,194 [INFO] Saving render image 5 to <PIL.Image.Image image mode=RGB size=256x256 at 0x230AE36B

In [16]:
# Display Render Video
Video(os.path.join(image_dir, "render.mp4"), embed=True)


In [50]:
import trimesh
import shutil
import os
from PIL import Image
from pygltflib import GLTF2, Image as GLTFImage, Texture, Sampler, Material, PbrMetallicRoughness, MeshPrimitive

# Load the .obj mesh using trimesh
mesh_trimesh = trimesh.load(mesh_file, process=False)

# Ensure texture image exists
texture_file = os.path.join(image_dir, "input_processed.png")

if os.path.exists(texture_file):
    # Open texture image properly
    texture_image = Image.open(texture_file)

    # Save a copy of the texture as PNG in the output directory
    texture_name = "texture.png"
    texture_path = os.path.join(image_dir, texture_name)
    texture_image.save(texture_path)

    # Assign texture to material
    material = trimesh.visual.material.SimpleMaterial(image=texture_path)
    
    # Ensure the mesh has a visual component and set texture
    mesh_trimesh.visual = trimesh.visual.TextureVisuals(material=material)

    # Convert .obj to GLTF using trimesh (with texture)
    glb_file = os.path.join(image_dir, "model_with_texture.glb")
    mesh_trimesh.export(glb_file)

    # Load the GLTF file using pygltflib to modify it
    gltf = GLTF2().load(glb_file)

    # Add texture to the GLTF file
    gltf_image = GLTFImage(uri=texture_name)
    gltf.images.append(gltf_image)

    # Add a sampler
    sampler = Sampler()
    gltf.samplers.append(sampler)

    # Create a texture reference
    texture = Texture(sampler=0, source=len(gltf.images) - 1)
    gltf.textures.append(texture)

    # Define material correctly
    material = Material(
        pbrMetallicRoughness=PbrMetallicRoughness(
            baseColorTexture={"index": len(gltf.textures) - 1}
        )
    )
    gltf.materials.append(material)

    # Assign the material to the mesh
    for mesh in gltf.meshes:
        for primitive in mesh.primitives:
            if isinstance(primitive, MeshPrimitive):  # Ensure correct primitive type
                primitive.material = len(gltf.materials) - 1

    # Save the modified .glb
    final_glb_file = os.path.join(image_dir, "model_final.glb")
    gltf.save(final_glb_file)

    # Ensure the texture file is copied
    shutil.copy(texture_file, os.path.join(image_dir, texture_name))

    logging.info(f"Mesh with texture exported as .glb to {final_glb_file}")

else:
    logging.warning("Texture file not found, exporting GLB without texture.")
    mesh_trimesh.export(os.path.join(image_dir, "model.glb"))

logging.info("GLB export complete.")


ImportError: cannot import name 'MeshPrimitive' from 'pygltflib' (c:\Users\akidu\anaconda3\envs\newenv\lib\site-packages\pygltflib\__init__.py)

In [ ]:
# import pymeshlab

# # Load the mesh using pymeshlab
# ms = pymeshlab.MeshSet()
# ms.load_new_mesh(mesh_file)

# # Access the current mesh
# mesh = ms.current_mesh()

# # Check if the mesh has texture coordinates (UVs)
# if not mesh.has_texcoord():
#     logging.info("Mesh has no UVs. Generating UV coordinates.")
#     ms.generate_parameterization_plumbing()  # Generate UV mapping

#     # Save the mesh with UV coordinates
#     mesh_with_uv_file = os.path.join(image_dir, "mesh_with_uv.obj")
#     ms.save_mesh(mesh_with_uv_file)
#     logging.info(f"Mesh with UV coordinates saved to {mesh_with_uv_file}")
# else:
#     logging.info("Mesh already has UV coordinates.")


AttributeError: 'pymeshlab.pmeshlab.Mesh' object has no attribute 'has_texcoord'

In [48]:
if mesh_trimesh.visual.uv is None:
    logging.error("Mesh has no UV coordinates! Texture will not be applied correctly.")
else:
    logging.info("Mesh has UV coordinates. Proceeding with texture mapping.")


2025-02-08 16:43:01,104 [ERROR] Mesh has no UV coordinates! Texture will not be applied correctly.
